# 15. Model Context Protocol (MCP) and Data Science Applications

**Difficulty:** Expert | **Time:** 3-4 hours | **Prerequisites:** Notebooks 01-14

By the end of this notebook you will be able to:

- Explain what MCP is and the problem it solves
- Describe MCP tools, resources, and prompts
- Compare LangChain tools with MCP tools
- Create a Data Science MCP server using FastMCP
- Connect a LangChain agent to MCP tools
- Understand MCP security considerations

---

## 1. What is the Model Context Protocol?

MCP is an **open protocol** that standardizes how AI applications connect to external tools and data.

### The Problem MCP Solves

Before MCP, every AI application needed custom integrations:

```
App A --> Custom Tool Integration 1
App B --> Custom Tool Integration 2
App C --> Custom Tool Integration 3
```

With MCP, there is a **standard interface**:

```
App A --+
App B --+--> MCP Protocol --> MCP Server --> Tools
App C --+                                --> Resources
                                         --> Prompts
```

### MCP Architecture

```mermaid
graph TD
    H[Host Application] --> MC[MCP Client]
    MC --> MS1[MCP Server 1]
    MC --> MS2[MCP Server 2]
    MS1 --> T1[Tools]
    MS1 --> R1[Resources]
    MS2 --> T2[Tools]
    MS2 --> P2[Prompts]
```

| Concept | Description | Analogy |
|---------|-------------|---------|
| **MCP Server** | Exposes tools, resources, prompts | A library of functions |
| **MCP Client** | Connects to servers, invokes tools | A library borrower |
| **Host Application** | The AI app that uses MCP | The user of the library |
| **Tools** | Functions the LLM can call | Buttons it can press |
| **Resources** | Data the LLM can read | Books it can read |
| **Prompts** | Reusable prompt templates | Templates it can fill in |

### Key insight

MCP provides **interoperability**. Write a tool once on an MCP server, and it works with Claude Desktop, LangChain agents, VS Code extensions, and any MCP-compatible client.

---

## 2. Setup


In [ ]:
import os
import json
import csv
import io
import re
from dotenv import load_dotenv
load_dotenv()

if os.getenv('OPENAI_API_KEY'):
    print('OpenAI API key found')
else:
    print('Warning: No OPENAI_API_KEY. Some examples will not work.')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
print('Core imports successful')

In [ ]:
# MCP imports
try:
    from fastmcp import FastMCP
    print('FastMCP imported')
except ImportError:
    print('FastMCP not installed. Run: pip install fastmcp')

try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    print('langchain-mcp-adapters imported')
except ImportError:
    print('langchain-mcp-adapters not installed. Run: pip install langchain-mcp-adapters')

In [ ]:
ollama_available = False
try:
    from langchain_ollama import ChatOllama
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    result = s.connect_ex(('127.0.0.1', 11434))
    s.close()
    if result == 0:
        ollama_available = True
        print('Ollama detected!')
    else:
        print('Ollama not running.')
except Exception:
    print('Ollama not available.')

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('LLM ready')

---

## 3. LangChain Tools vs MCP Tools

| Aspect | LangChain Tool | MCP Tool |
|--------|---------------|----------|
| **Definition** | `@tool` decorator | `@mcp.tool()` decorator |
| **Transport** | In-process call | JSON-RPC over stdio or HTTP |
| **Interoperability** | LangChain only | Any MCP client |
| **Server model** | Tools in application | Tools on separate server |
| **Discovery** | Passed directly | Discovered at runtime |

### Why MCP matters

- **Separation of concerns**: Tool logic lives in the server
- **Reusability**: Same server works with Claude, LangChain, VS Code, etc.
- **Standardization**: One protocol, many clients
- **Security**: Server controls what tools are exposed

In [ ]:
# LangChain tool (what we have been using)
@tool
def langchain_add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# MCP tool uses the same logic but runs on a separate server
print('LangChain tool:', langchain_add.name)
print('Description:', langchain_add.description)

---

## 4. Creating a Data Science MCP Server

We use **FastMCP** to create MCP servers. Here is the server code:

```python
# examples/mcp/ds_mcp_server.py
from fastmcp import FastMCP
import json
import statistics
import csv
import io

mcp = FastMCP('DataScienceTools')

@mcp.tool()
def calculate_mean(numbers: str) -> str:
    """Calculate the mean of comma-separated numbers."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    return json.dumps({'mean': statistics.mean(nums), 'count': len(nums)})

@mcp.tool()
def calculate_stdev(numbers: str) -> str:
    """Calculate the standard deviation."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    return json.dumps({'stdev': statistics.stdev(nums)})

@mcp.tool()
def dataset_summary(data: str) -> str:
    """Summarize a CSV dataset."""
    reader = csv.DictReader(io.StringIO(data))
    rows = list(reader)
    return json.dumps({'rows': len(rows), 'columns': list(rows[0].keys())})

if __name__ == '__main__':
    mcp.run(transport="stdio")
```

In [ ]:
# Demonstrate the tool logic directly
import statistics

def calculate_mean(numbers_str):
    nums = [float(x.strip()) for x in numbers_str.split(',')]
    return {'mean': statistics.mean(nums), 'count': len(nums)}

def calculate_stdev(numbers_str):
    nums = [float(x.strip()) for x in numbers_str.split(',')]
    if len(nums) < 2:
        return {'error': 'Need at least 2 numbers'}
    return {'stdev': statistics.stdev(nums), 'count': len(nums)}

print('calculate_mean("10, 20, 30, 40, 50"):', calculate_mean('10, 20, 30, 40, 50'))
print('calculate_stdev("10, 20, 30, 40, 50"):', calculate_stdev('10, 20, 30, 40, 50'))

---

## 5. How MCP Works (Protocol Details)

MCP uses **JSON-RPC 2.0** for client-server communication.

### Message flow

```
Client -> Server:  initialize
Server -> Client:  capabilities (tools, resources, prompts)
Client -> Server:  tools/list
Server -> Client:  list of tools with schemas
Client -> Server:  tools/call {name: 'calculate_mean', arguments: {...}}
Server -> Client:  tool result
```

### Transport types

| Transport | Description | Use case |
|-----------|-------------|----------|
| **stdio** | Standard input/output | Local tools, simple setup |
| **streamable-http** | HTTP requests | Remote servers, web services |

---

## 6. Connecting LangChain to MCP Tools

`langchain-mcp-adapters` bridges MCP and LangChain.

In [ ]:
# Convert MCP-style tools to LangChain tools
# (Simulates what the MCP adapter does automatically)

@tool
def mcp_calculate_mean(numbers: str) -> str:
    """Calculate the mean of comma-separated numbers like '10, 20, 30'."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    return json.dumps({'mean': statistics.mean(nums), 'count': len(nums)})

@tool
def mcp_calculate_stdev(numbers: str) -> str:
    """Calculate the standard deviation of comma-separated numbers."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    if len(nums) < 2:
        return json.dumps({'error': 'Need at least 2 numbers'})
    return json.dumps({'stdev': statistics.stdev(nums), 'count': len(nums)})

tools = [mcp_calculate_mean, mcp_calculate_stdev]
tool_desc = chr(10).join(['- ' + t.name + ': ' + t.description for t in tools])
print('Tools available:')
print(tool_desc)

In [ ]:
# Build agent with MCP-style tools
from langchain_core.messages import HumanMessage

agent_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Data Science assistant. Tools available:\n' + tool_desc + '\n\nTo use a tool, call it directly. After receiving the result, provide a natural language answer.'),
    ('human', '{input}')
])
agent_chain = agent_prompt | llm | StrOutputParser()

response = agent_chain.invoke({'input': 'Calculate the mean of 15, 25, 35, 45'})
print('Agent response:')
print(response[:300])

---

## 7. MCP with LangGraph (Modern Pattern)

The modern approach uses `create_agent` with MCP tools:

```python
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

async def main():
    client = MultiServerMCPClient({
        'ds_tools': {
            'transport': 'stdio',
            'command': 'python',
            'args': ['ds_mcp_server.py']
        }
    })
    tools = await client.get_tools()
    agent = create_agent('openai:gpt-4o-mini', tools)
    result = await agent.ainvoke({
        'messages': [{'role': 'user', 'content': 'Calculate the mean of 10, 20, 30'}]
    })
    print(result)

asyncio.run(main())
```

### Multiple servers

```python
client = MultiServerMCPClient({
    'math': {'transport': 'stdio', 'command': 'python', 'args': ['math_server.py']},
    'database': {'transport': 'http', 'url': 'http://localhost:8000/mcp'},
    'analytics': {'transport': 'stdio', 'command': 'python', 'args': ['analytics_server.py']}
})
tools = await client.get_tools()  # Tools from ALL servers
```

---

## 8. MCP Resources and Prompts

| Feature | Purpose | Example |
|---------|---------|---------|
| **Tools** | Execute actions | Calculate statistics |
| **Resources** | Provide readable data | Dataset documentation |
| **Prompts** | Reusable templates | Analysis template |

In [ ]:
# MCP Resources: data exposed by the server
dataset_docs = {
    'students': {'description': 'Student performance', 'columns': ['name', 'age', 'grade', 'score'], 'rows': 100},
    'sales': {'description': 'Monthly sales', 'columns': ['product', 'quantity', 'price', 'date'], 'rows': 500}
}
print('MCP Resources: Dataset Documentation')
print(json.dumps(dataset_docs, indent=2))

In [ ]:
# MCP Prompts: reusable templates from the server
mcp_prompts = {
    'analyze_dataset': 'Analyze dataset {dataset_name}: 1) Summary stats, 2) Quality issues, 3) Visualizations',
    'compare_groups': 'Compare {group_a} vs {group_b} on variable {variable}',
    'explain_metric': 'Explain {metric} in {domain}: definition, interpretation, when to use'
}
print('MCP Prompts available:')
for name, template in mcp_prompts.items():
    print(f'  {name}: {template}')

---

## 9. Complete Data Science MCP Workflow


In [ ]:
# Sample data
students_csv = 'name,age,grade,score\nAlice,20,A,92\nBob,21,B,78\nCarol,19,A,95\nDavid,22,C,65\nEve,20,B,82\nFrank,21,A,88\nGrace,19,B,76\nHenry,23,C,58'

def dataset_summary(csv_data):
    reader = csv.DictReader(io.StringIO(csv_data))
    rows = list(reader)
    columns = list(rows[0].keys())
    numeric_cols = []
    for col in columns:
        try:
            float(rows[0][col])
            numeric_cols.append(col)
        except (ValueError, KeyError):
            pass
    summary = {'rows': len(rows), 'columns': columns, 'numeric_columns': numeric_cols}
    for col in numeric_cols:
        values = [float(r[col]) for r in rows]
        summary[col] = {'mean': round(statistics.mean(values), 2), 'min': min(values), 'max': max(values)}
    return summary

print('Dataset Summary:')
print(json.dumps(dataset_summary(students_csv), indent=2))

In [ ]:
# Complete agent with all tools
@tool
def ds_summary(csv_data: str) -> str:
    """Summarize a CSV dataset. Input: CSV-formatted string."""
    return json.dumps(dataset_summary(csv_data))

@tool
def ds_mean(numbers: str) -> str:
    """Calculate mean of comma-separated numbers."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    return json.dumps({'mean': statistics.mean(nums)})

@tool
def ds_stdev(numbers: str) -> str:
    """Calculate standard deviation of comma-separated numbers."""
    nums = [float(x.strip()) for x in numbers.split(',')]
    return json.dumps({'stdev': statistics.stdev(nums) if len(nums) > 1 else 0})

all_tools = [ds_summary, ds_mean, ds_stdev]
tool_desc2 = chr(10).join(['- ' + t.name + ': ' + t.description for t in all_tools])

ds_agent_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Data Science assistant with MCP tools:\n' + tool_desc2 + '\nUse tools when calculations are needed.'),
    ('human', '{input}')
])
ds_agent = ds_agent_prompt | llm | StrOutputParser()

response = ds_agent.invoke({
    'input': 'Here is a dataset:\n' + students_csv + '\n\nSummarize this dataset and calculate the standard deviation of scores.'
})
print('DS Agent response:')
print(response[:500])

---

## 10. Local Ollama with MCP

MCP is protocol-level. The same server works with any LLM provider.

```python
# MCP server stays the same. Only the model changes:
from langchain_ollama import ChatOllama
ollama_llm = ChatOllama(model='llama3.2')
agent = create_agent(ollama_llm, tools)  # Same MCP tools
```

In [ ]:
if ollama_available:
    ollama_llm = ChatOllama(model='llama3.2', temperature=0)
    ollama_agent_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a Data Science assistant. Tools:\n' + tool_desc2),
        ('human', '{input}')
    ])
    ollama_agent = ollama_agent_prompt | ollama_llm | StrOutputParser()
    response = ollama_agent.invoke({'input': 'Calculate mean and stdev of 5, 10, 15, 20, 25'})
    print('Ollama MCP agent:', response[:300])
else:
    print('Ollama not available. Start with: ollama serve')

---

## 11. MCP Security Considerations

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Malicious servers** | Server exposes dangerous tools | Only connect to trusted servers |
| **Untrusted tools** | Tool performs unexpected actions | Validate inputs, audit behavior |
| **Data leakage** | Sensitive data sent externally | Use local servers for sensitive data |
| **Tool injection** | Server manipulates LLM via output | Validate and sanitize outputs |
| **Authentication** | Unauthorized access | Use HTTP auth, API keys |

### Secure MCP architecture

```mermaid
graph LR
    A[LLM App] --> B[MCP Client]
    B --> C{Trust Check}
    C -->|trusted| D[MCP Server]
    C -->|untrusted| E[Blocked]
    D --> F[Input Validation]
    F --> G[Tool Execution]
    G --> H[Output Validation]
    H --> I[Result]
```

### Best practices

1. Only connect to MCP servers you trust
2. Validate all tool inputs and outputs
3. Use authentication for HTTP servers
4. Monitor tool calls for unusual patterns
5. Use local servers for sensitive data

In [ ]:
# Security example: validated tool wrapper
class SecureMCPToolWrapper:
    def __init__(self, tool_fn, allowed_patterns=None):
        self.tool_fn = tool_fn
        self.allowed_patterns = allowed_patterns or [r'^[0-9,\.\s-]+$']
        self.log = []
    
    def invoke(self, input_text):
        valid = any(re.match(p, input_text.strip()) for p in self.allowed_patterns)
        if not valid:
            self.log.append({'input': input_text, 'status': 'BLOCKED'})
            return json.dumps({'error': 'Input failed validation'})
        result = self.tool_fn(input_text)
        self.log.append({'input': input_text[:50], 'status': 'OK'})
        return result
    
    def summary(self):
        print(f'Calls: {len(self.log)}')
        for e in self.log:
            print(f'  [{e["status"]}] {e["input"]}')

secure_calc = SecureMCPToolWrapper(calculate_mean)
print(secure_calc.invoke('10, 20, 30'))
print(secure_calc.invoke(' DROP TABLE students; --'))  # Blocked!
secure_calc.summary()

---

## 12. Exercises

### Exercise 1: Add a New MCP Tool
Add a `calculate_correlation` tool that computes correlation between two number lists.

### Exercise 2: Build an MCP Resource Server
Create an MCP server exposing 3 dataset descriptions as resources.

### Exercise 3: MCP Security Audit
Review the MCP server code and implement mitigations for all security issues.

### Challenge: Multi-Server DS Assistant
Design two MCP servers: Statistics Server and Data Server. Connect both to one agent.

### Challenge: Visualization MCP Server
Create an MCP server that generates chart specifications from data.

---

## 13. Key Takeaways

| Concept | Key Point |
|---------|-----------|
| **MCP** | Open standard for connecting AI apps to tools and data |
| **Tools** | Functions the LLM can call via JSON-RPC |
| **Resources** | Readable data exposed by MCP servers |
| **Prompts** | Reusable prompt templates from MCP servers |
| **FastMCP** | Python library for creating MCP servers easily |
| **langchain-mcp-adapters** | Bridges MCP tools to LangChain |
| **Interoperability** | Same server works with any MCP client |
| **Security** | Trust servers, validate inputs/outputs, use auth |

### The complete 15-notebook stack

| # | Notebook | Core Skill |
|---|----------|------------|
| 01 | Introduction | LangChain basics |
| 02 | Models, Prompts, Messages | LLM interaction |
| 03 | LCEL and Chains | Pipeline composition |
| 04 | Embeddings and Vector Stores | Semantic search |
| 05 | RAG | Knowledge-grounded generation |
| 06 | Tools and Agents | Dynamic workflows |
| 07 | Capstone Project | Complete application |
| 08 | Advanced RAG | Production RAG techniques |
| 09 | Document Loading | Multi-format processing |
| 10 | SQL and Databases | Structured data interaction |
| 11 | Data Science Agents | Agent-based analysis |
| 12 | LangGraph | Stateful graph workflows |
| 13 | Evaluation | Testing and observability |
| 14 | Security | Defense and mitigation |
| 15 | MCP | Standardized tool protocol |